# 03. Feature Engineering - Clinical Predictors & Departmental Aggregations
**Author:** Ravikant Yadav  
**Date:** June 23, 2026  

In hospital administration, granular transactional details must be transformed into operational features. In this notebook, we engineer:
- Patient Length of Stay (LOS) in days
- Clinical treatment density and accumulated costs
- Financial write-off risk flags
- Consolidated departmental and doctor caseload indicators


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "cleaned"

admissions = pd.read_csv(DATA_DIR / "admissions.csv")
billing = pd.read_csv(DATA_DIR / "billing.csv")
treatments = pd.read_csv(DATA_DIR / "treatments.csv")
doctors = pd.read_csv(DATA_DIR / "doctors.csv")


## 1. Feature 1: Clinical Length of Stay (LOS)
Length of Stay is a vital driver of hospital operational efficiency. Let's calculate and verify this metric.


In [ ]:
admissions['admission_date'] = pd.to_datetime(admissions['admission_date'])
admissions['discharge_date'] = pd.to_datetime(admissions['discharge_date'])

# Stay duration in days
calculated_los = (admissions['discharge_date'] - admissions['admission_date']).dt.days
admissions['length_of_stay_calculated'] = calculated_los.fillna(1) # default to 1 day if active stay

print("--- Calculated Length of Stay (LOS) ---")
print(admissions[['admission_id', 'admission_date', 'discharge_date', 'length_of_stay_calculated']].head())


## 2. Feature 2: Procedure Cost Accumulation & Density
Let's aggregate clinical procedures and total procedural costs per patient stay from `treatments.csv`.


In [ ]:
print("Treatments columns:", treatments.columns.tolist())

# Aggregate by admission_id
treatment_agg = treatments.groupby('admission_id').agg(
    total_procedure_procedures=('procedure_code', 'count'),
    accumulated_treatment_cost=('treatment_cost', 'sum'),
    avg_procedure_cost=('treatment_cost', 'mean')
).reset_index()

print("--- Engineered Procedure Cost Features ---")
print(treatment_agg.head())


## 3. Feature 3: Financial Out-of-Pocket Risk Flag
Let's calculate outstanding write-off balances and flag cases with high default risk (where insurance coverage is low).


In [ ]:
billing['outstanding_patient_balance'] = billing['charge_amount'] - billing['insurance_paid'] - billing['patient_paid']
billing['high_write_off_risk'] = (billing['outstanding_patient_balance'] > 5000).astype(int)

print("--- Engineered Financial Risk Flags ---")
print(billing[['admission_id', 'outstanding_patient_balance', 'high_write_off_risk']].head())


## 4. Feature 4: Consolidated Clinical Feature Matrix
Now, we merge our patient demographic features, length of stay, procedure counts, and billing risks into a single unified analytical dataset.


In [ ]:
# Merge operations
master_df = admissions[['admission_id', 'patient_id', 'doctor_id', 'department_id', 'admission_type', 'wait_minutes', 'severity_score', 'readmission_30d', 'discharge_efficiency_score']].copy()
master_df = pd.merge(master_df, treatment_agg, on='admission_id', how='left')
master_df = pd.merge(master_df, billing[['admission_id', 'charge_amount', 'cost_amount', 'high_write_off_risk']], on='admission_id', how='left')

# Fill missing procedure aggregations
master_df['total_procedure_procedures'] = master_df['total_procedure_procedures'].fillna(0)
master_df['accumulated_treatment_cost'] = master_df['accumulated_treatment_cost'].fillna(0.0)
master_df['avg_procedure_cost'] = master_df['avg_procedure_cost'].fillna(0.0)

print("\n--- Master Hospital Feature Matrix Completed ---")
print(master_df.info())
print(master_df.describe().T)
